#03 - Silver to Gold
Criação do Star Schema para consumo de BI, geração de tabelas-ponte e consolidação do documento de contexto para alimentar o Banco de Dados Vetorial do time de IA.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Parâmetros de Padronização
catalog = "workspace"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

# Checks de boas práticas para Data Quality
dq_results = []

def dq_check_condition(table_name: str, check_name: str, df, condition_expr: str):
    total = df.count()
    failed = df.filter(~F.expr(condition_expr)).count()
    passed = (failed == 0)
    dq_results.append((table_name, check_name, total, failed, "PASS" if passed else "FAIL", datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {failed}/{total} falhas")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = (dupes == 0)
    dq_results.append((table_name, check_name, total, dupes, "PASS" if passed else "FAIL", datetime.now()))

In [0]:
# gold.dim_movies
df_info = spark.table(f"{silver_schema}.tb_info_filmes")

# Utilização de hash SHA2 para gerar Surrogate Keys
df_dim_movies = (
    df_info
    .withColumn("sk_movie_id", F.expr("conv(substr(hex(sha2(id_filme, 256)), 1, 16), 16, 10)").cast("bigint"))
    .select(
        "sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento", 
        "duracao_minutos", "idioma_original", "status_filme", "sinopse"
    )
)

dq_check_unique("dim_movies", "pk_unica", df_dim_movies, ["sk_movie_id"])
df_dim_movies.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.dim_movies")

# gold.dim_reviews
df_reviews_silver = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")

df_dim_reviews = (
    df_reviews_silver
    .groupBy("id_filme")
    .agg(
        F.count("comentario_usuario").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
    )
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), "id_filme", "inner")
    .withColumn("sk_review_id", F.expr("conv(substr(hex(sha2(cast(sk_movie_id as string), 256)), 1, 16), 16, 10)").cast("bigint"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)

dq_check_unique("dim_reviews", "pk_unica", df_dim_reviews, ["sk_review_id"])
df_dim_reviews.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.dim_reviews")

In [0]:
# gold.dim_genres
df_genres_silver = spark.table(f"{silver_schema}.tb_generos").select("nome_genero").distinct()

df_dim_genres = (
    df_genres_silver
    .withColumn("sk_genre_id", F.expr("conv(substr(hex(sha2(nome_genero, 256)), 1, 16), 16, 10)").cast("bigint"))
    .select("sk_genre_id", "nome_genero")
)
dq_check_unique("dim_genres", "pk_unica", df_dim_genres, ["sk_genre_id"])
df_dim_genres.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.dim_genres")

# gold.dim_people
df_pessoas_silver = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select("nome_entidade", "tipo_entidade").distinct()
)

df_dim_people = (
    df_pessoas_silver
    .withColumnRenamed("nome_entidade", "nome_pessoa")
    .withColumnRenamed("tipo_entidade", "tipo_pessoa")
    .withColumn("sk_person_id", F.expr("conv(substr(hex(sha2(concat(nome_pessoa, tipo_pessoa), 256)), 1, 16), 16, 10)").cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)
dq_check_unique("dim_people", "pk_unica", df_dim_people, ["sk_person_id"])
df_dim_people.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.dim_people")

# gold.dim_companies
df_companies_silver = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select("nome_entidade").distinct()
)

df_dim_companies = (
    df_companies_silver
    .withColumnRenamed("nome_entidade", "nome_produtora")
    .withColumn("sk_company_id", F.expr("conv(substr(hex(sha2(nome_produtora, 256)), 1, 16), 16, 10)").cast("bigint"))
    .select("sk_company_id", "nome_produtora")
)
dq_check_unique("dim_companies", "pk_unica", df_dim_companies, ["sk_company_id"])
df_dim_companies.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.dim_companies")

In [0]:

# Criação das Bridge Tables
df_filmes_ref = df_dim_movies.select("sk_movie_id", "id_filme")

# Bridge Genres
df_bridge_genre = (
    spark.table(f"{silver_schema}.tb_generos")
    .join(df_filmes_ref, "id_filme", "inner")
    .join(df_dim_genres, "nome_genero", "inner")
    .select("sk_movie_id", "sk_genre_id")
)
df_bridge_genre.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.bridge_movie_genre")

# Bridge People
df_bridge_person = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .withColumnRenamed("nome_entidade", "nome_pessoa")
    .withColumnRenamed("tipo_entidade", "tipo_pessoa")
    .join(df_filmes_ref, "id_filme", "inner")
    .join(df_dim_people, ["nome_pessoa", "tipo_pessoa"], "inner")
    .select("sk_movie_id", "sk_person_id")
)
df_bridge_person.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.bridge_movie_person")

# Bridge Companies
df_bridge_company = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .withColumnRenamed("nome_entidade", "nome_produtora")
    .join(df_filmes_ref, "id_filme", "inner")
    .join(df_dim_companies, "nome_produtora", "inner")
    .select("sk_movie_id", "sk_company_id")
)
df_bridge_company.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.bridge_movie_company")

In [0]:
# gold.fact_movies_performance
# Join exclusivo de filmes 'Lançados' consolidando financeiro e métricas
df_fin = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_met = spark.table(f"{silver_schema}.tb_metricas_engajamento")

df_fact = (
    df_dim_movies.filter(F.col("status_filme") == "Lançado").select("sk_movie_id", "id_filme")
    .join(df_fin, "id_filme", "left")
    .join(df_met, "id_filme", "left")
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").cast("decimal(18,2)"), 
        F.col("receita_usd").cast("decimal(18,2)"), 
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"), 
        F.col("receita_brl").cast("decimal(18,2)"), 
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"), 
        F.col("nota_media_tmdb").cast("double"), 
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"), 
        F.col("qtd_votos_imdb").cast("int")
    )
)

dq_check_unique("fact_movies_performance", "pk_unica", df_fact, ["sk_movie_id"])
df_fact.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.fact_movies_performance")

In [0]:
# gold_genai_movies_context (Preparação RAG para LLM)

# Agrupando atores em uma única string por filme
df_atores_agg = (
    df_bridge_person
    .join(df_dim_people.filter("tipo_pessoa = 'Ator'"), "sk_person_id")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.collect_list("nome_pessoa")).alias("atores_principais"))
)

# Agrupando diretores
df_diretores_agg = (
    df_bridge_person
    .join(df_dim_people.filter("tipo_pessoa = 'Diretor'"), "sk_person_id")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(" e ", F.collect_list("nome_pessoa")).alias("diretor"))
)

df_context_base = (
    df_dim_movies.select("sk_movie_id", "id_filme", "titulo", "ano_lancamento", "sinopse")
    .join(spark.table(f"{gold_schema}.fact_movies_performance").select("sk_movie_id", "receita_usd", "orcamento_usd"), "sk_movie_id", "left")
    .join(df_atores_agg, "sk_movie_id", "left")
    .join(df_diretores_agg, "sk_movie_id", "left")
)

# Tratamento Anti-Nulo para Concatenação de GenAI
df_genai = df_context_base.withColumn(
    "llm_context_document",
    F.concat_ws(" ",
        F.lit("O filme"), 
        F.coalesce(F.col("titulo"), F.lit("desconhecido")),
        F.lit("lançado no ano de"), 
        F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("não informado")),
        F.lit("faturou"), 
        F.coalesce(F.concat(F.lit("US$ "), F.col("receita_usd").cast("string")), F.lit("um valor não divulgado")),
        F.lit("e teve um custo de"), 
        F.coalesce(F.concat(F.lit("US$ "), F.col("orcamento_usd").cast("string")), F.lit("orçamento desconhecido.")),
        F.lit("Estrelado por"), 
        F.coalesce(F.col("atores_principais"), F.lit("elenco não creditado")),
        F.lit("e dirigido por"), 
        F.coalesce(F.col("diretor"), F.lit("diretor não especificado")),
        F.lit(", o filme possui a seguinte sinopse:"),
        F.coalesce(F.col("sinopse"), F.lit("Sinopse indisponível."))
    )
).select(
    F.col("id_filme").alias("movie_id"),
    F.col("titulo").alias("title"),
    F.col("llm_context_document")
)

df_genai.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.gold_genai_movies_context")

display(df_genai.limit(2))

movie_id,title,llm_context_document
584007,"Однажды в Америке, или Чисто русская сказка","O filme Однажды в Америке, или Чисто русская сказка lançado no ano de 2019 faturou um valor não divulgado e teve um custo de orçamento desconhecido. Estrelado por Mikhail Bashkatov, Anatoliy Kalmykov, Nikolay Bandurin, Marina Orlova, Anna Partseva, Valeriy Ponomarenko, Emmanuil Vitorgan, Aleksandr Ponomarenko, Aslan Bizhoyev, Viktoriya Voronova e dirigido por Dmitri Panchenko , o filme possui a seguinte sinopse: Sinopse indisponível."
482997,★,"O filme ★ lançado no ano de 2017 faturou um valor não divulgado e teve um custo de orçamento desconhecido. Estrelado por elenco não creditado e dirigido por Johann Lurf , o filme possui a seguinte sinopse: The stars in the night's sky. Pinpricks of light against the darkness excerpted from films beginning at cinema's dawn and continuing to this present day in a project that is planned to be expanded yearly."


In [0]:
# Validação da Camada Gold
# Resolução das 6 perguntas de negócio utilizando Spark SQL para validar a integridade do *Star Schema*.

# 1. Qual é a receita total (em R$) somada de todos os filmes da base?
display(spark.sql(f"""
    SELECT 
        CAST(SUM(receita_brl) AS DECIMAL(20,2)) AS receita_total_brl
    FROM {gold_schema}.fact_movies_performance
"""))

# 2. Quais são os 5 filmes com maior popularidade?
display(spark.sql(f"""
    SELECT 
        m.titulo, 
        f.popularidade 
    FROM {gold_schema}.dim_movies m
    JOIN {gold_schema}.fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
    ORDER BY f.popularidade DESC NULLS LAST
    LIMIT 5
"""))

# 3. Quantos filmes cada gênero possui? (Do maior para o menor)
display(spark.sql(f"""
    SELECT 
        g.nome_genero, 
        COUNT(DISTINCT b.sk_movie_id) as qtd_filmes
    FROM {gold_schema}.dim_genres g
    JOIN {gold_schema}.bridge_movie_genre b ON g.sk_genre_id = b.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC
"""))

# 4. Para os 10 filmes de maior receita, mostre título, receita (US$ e R$) e a posição no ranking.
display(spark.sql(f"""
    SELECT 
        m.titulo, 
        f.receita_usd, 
        f.receita_brl, 
        RANK() OVER (ORDER BY f.receita_usd DESC) as ranking
    FROM {gold_schema}.dim_movies m
    JOIN {gold_schema}.fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
    ORDER BY ranking
    LIMIT 10
"""))

# 5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos?
display(spark.sql(f"""
    WITH DataLimite AS (
        -- Descobre a data de lançamento mais recente válida
        SELECT MAX(data_lancamento) as max_data
        FROM {gold_schema}.dim_movies
        WHERE data_lancamento <= CURRENT_DATE()
    )
    SELECT 
        p.nome_pessoa, 
        COUNT(DISTINCT b.sk_movie_id) as qtd_participacoes
    FROM {gold_schema}.dim_people p
    JOIN {gold_schema}.bridge_movie_person b ON p.sk_person_id = b.sk_person_id
    JOIN {gold_schema}.dim_movies m ON b.sk_movie_id = m.sk_movie_id
    CROSS JOIN DataLimite dl
    WHERE p.tipo_pessoa = 'Ator'
      -- Filtro de 2 anos a partir da data máxima encontrada
      AND m.data_lancamento >= add_months(dl.max_data, -24)
    GROUP BY p.nome_pessoa
    ORDER BY qtd_participacoes DESC
    LIMIT 1
"""))

# 6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos?
display(spark.sql(f"""
    WITH DataLimite AS (
        SELECT MAX(data_lancamento) as max_data
        FROM {gold_schema}.dim_movies
        WHERE data_lancamento <= CURRENT_DATE()
    )
    SELECT 
        c.nome_produtora, 
        CAST(SUM(f.lucro_usd) AS DECIMAL(20,2)) as lucro_total_usd
    FROM {gold_schema}.dim_companies c
    JOIN {gold_schema}.bridge_movie_company b ON c.sk_company_id = b.sk_company_id
    JOIN {gold_schema}.dim_movies m ON b.sk_movie_id = m.sk_movie_id
    JOIN {gold_schema}.fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
    CROSS JOIN DataLimite dl
    WHERE m.data_lancamento >= add_months(dl.max_data, -60)
    GROUP BY c.nome_produtora
    ORDER BY lucro_total_usd DESC NULLS LAST
    LIMIT 1
"""))

receita_total_brl
864695614124.59


titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
retribution,1547.22


nome_genero,qtd_filmes
Drama,29554
Documentary,18324
Comedy,16943
Thriller,9165
Horror,8848
Romance,6893
Action,5394
Crime,4221
Animation,4001
Science Fiction,3364


titulo,receita_usd,receita_brl,ranking
Avengers: Endgame,2800000000.00,14439320000.00,1
Avatar: The Way of Water,2320250281.00,11965298674.09,2
AVENGERS: INFINITY WAR,2052415039.00,10584099114.62,3
spider-man: no way home,1921847111.00,9910773366.72,4
The Lion King,1663075401.00,8576313535.42,5
Top Gun: Maverick,1488732821.00,7677246284.61,6
Barbie,1428545028.00,7366863854.89,7
The Super Mario Bros. Movie,1355725263.00,6991339608.76,8
Black Panther,1349926083.00,6961433817.42,9
Star Wars: The Last Jedi,1332698830.00,6872594596.43,10


nome_pessoa,qtd_participacoes
Eric Roberts,5


nome_produtora,lucro_total_usd
Universal Pictures,6181119834.00


In [0]:
# Validação completa e comparativa da Tabela Fato com os títulos da Dimensão
display(
    spark.sql(f"""
        SELECT 
            m.id_filme,
            m.titulo,
            f.sk_movie_id,
            f.orcamento_usd,
            f.receita_usd,
            f.lucro_usd,
            f.orcamento_brl,
            f.receita_brl,
            f.lucro_brl
        FROM {gold_schema}.fact_movies_performance f
        JOIN {gold_schema}.dim_movies m ON f.sk_movie_id = m.sk_movie_id
        ORDER BY CAST(m.id_filme AS INT) ASC
        LIMIT 20
    """)
)

id_filme,titulo,sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl
14564,Rings,7076671461002470196,25000000.00,83080890.00,58080890.00,128922500.00,428439841.64,299517341.64
32471,Mixtape,7219894958922413881,null,null,null,null,null,null
38258,Grizzly II: Revenge,3833800660322957620,7.50,null,null,38.68,null,null
38492,Billy Joel - Live at Yankee Stadium,4049361009342899251,null,null,null,null,null,null
38700,Bad Boys for Life,3834590998028236642,90.00,426505244.00,426505154.00,464.12,2199444892.78,2199444428.66
42018,The Horse Thief,7291669990602192948,null,null,null,null,null,null
42330,Monkey Magic,4121419712358070071,null,null,null,null,null,null
43074,Ghostbusters,3990533650582681655,144000000.00,229147509.00,85147509.00,742593600.00,1181690789.16,439097189.16
45033,20 Seconds of Joy,3846467030013666917,337200.00,null,null,1738906.68,null,null
46983,The Song of Styrene,3762023432425006392,null,null,null,null,null,null
